# State-Specific Baseline Upfront Solar Cost — LBNL Tracking the Sun (2026 release)

**Purpose.** Produce the per-state **baseline upfront residential PV cost ($/Wdc)** that the dGen
run uses, from LBNL's published state-level median installed prices, and **document every
methodology decision with the evidence behind it** so the choices are auditable later.

**Output:** `data/state_upfront_cost_lbnl_2025.csv` — one row per state (+ `National`), columns
`state, upfront_cost_per_w, source, n, lbnl_median_per_w`.

### Logic flow
```
tts_state_download_data_2026_release.xlsx  (LBNL PUBLISHED state medians, "Data" tab)
      │  filter: Type=Residential, year=2025, metric=price_dollar_per_watt_median
      ▼
  per-state median $/Wdc  +  price_N (sample size)
      │  rule: use own median if price_N >= 100, else National median ($3.62)
      ▼
  data/state_upfront_cost_lbnl_2025.csv                         ← THIS NOTEBOOK
      │
      ▼
  adjust_pv_batt_price_trajectories.ipynb  →  make_state_specific()
      │  anchor each state's 2026 capex to its $/W, apply national ATB decline shape
      │  build BOTH pv_price_baseline AND pv_plus_batt_baseline (state-keyed) → upload to Cloud SQL
      ▼
  dGen run: agent_mutation.elec.apply_pv_prices / apply_pv_plus_batt_prices
            merge on state_abbr → financial_functions economics & adoption
```

## Decision 1 — Use LBNL's PUBLISHED state-median table, not the row-level public file

**Decision:** source medians from `tts_state_download_data_2026_release.xlsx` (LBNL's aggregated
"Tracking the Sun" state download), **not** from `TTS_LBNL_public_file_*.csv` (the row-level file).

**Why / evidence.** The row-level public file is a *partial extract* and cannot reproduce LBNL's
published medians:

| Evidence | Row-level public file | LBNL published / viz tool |
|---|---|---|
| NJ priced records (all years) | 230,116 rows, **0 with a usable price** (100% redacted) | NJ median **$3.74** |
| CT 2024 records (all segments) | **2,742** total | tool shows **3,663** — file can't contain them |
| NY 2024 (LBNL-method priced) | 7,745 | tool shows 9,385 |
| States covered | **27** | 51 + National |
| National median (pv-only, host-owned) | 2024 $3.92 / 2025 **$3.51** | 2024 $4.09 / 2025 **$3.62** |

The ~3–4% national gap is **state-dependent** (CA matches within 1.5%, NY is ~5% low, NJ is missing
entirely), so no single scaling factor fixes it. The published aggregated table *is* LBNL's full
dataset and matches their viz tool exactly (verified in the sanity-check cell below), so we use it
directly.

## Decision 2 — Metric: PV-only, host-owned, non-self-installed, gross price

**Decision:** use `price_dollar_per_watt_median` — the **median installed price ($/Wdc)**, the
**gross** price paid by the owner *before* incentives/tax credits.

Per LBNL's published definition: *"Installed price represents the gross price paid by the system
owner, prior to receipt of incentives or tax credits. Trends shown exclude third-party-owned and
self-installed systems."* The published table already applies this (TPO/self-installed excluded).

**Why PV-only (evidence):** on the row-level file we tested including paired storage. Including
`pv+storage` (total price ÷ PV watts) **flattens the year-over-year trend** — national goes
$4.25→$4.19 (2024→2025) instead of the real ~11% decline ($3.92→$3.51 pv-only). The published
$/W trend shows the decline, confirming it is a **PV-only** price series (storage cost must not be
folded into $/Wdc). Units are DC nameplate — consistent with dGen's `system_capex_per_kw`
(the model sizes and prices on the DC side; `ac = dc * 0.96` is applied only to energy).

## Decision 3 — Year = 2025; Decision 4 — sufficiency bar n ≥ 100; Decision 5 — national fallback

- **Year = 2025.** Most recent year; confirmed by LBNL to be **complete** (not still filling in).
  National 2025 residential median = **$3.6159/W**.
- **Sufficiency: a state uses its own median only if `price_N >= 100`.** Below that the sample is
  too thin to trust. This sends **MD ($3.95, n=29), NH ($3.65, n=43), MA ($3.37, n=26)** to the
  national value despite having a published median — a deliberate choice for robustness.
- **Fallback = the National 2025 median ($3.62)** for every state without a sufficient own median
  (thin sample *or* no data). 13 states clear the bar; 38 use the fallback.

**Battery cost note (set in the trajectory notebook, documented here):** battery $/kWh stays a
**2024** value ($1,199.3/kWh) on purpose. LBNL publishes no aggregated 2025 storage $/kWh, and the
row-level 2025 storage price is too noisy to use (RES-only ~$2,077/kWh on n=177; RES+RES_SF ~$1,549,
a ~28% jump that is a data-mix artifact, not real cost inflation). Only PV cost is made
state-specific and updated to 2025.

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = os.path.abspath(os.path.join(os.path.abspath("."), "..", "..", "..", "data"))
SRC_XLSX = os.path.join(DATA_DIR, "tts_state_download_data_2026_release.xlsx")
OUT_CSV  = os.path.join(DATA_DIR, "state_upfront_cost_lbnl_2025.csv")

YEAR       = 2025            # most recent complete year (Decision 3)
TYPE       = "Residential"
MIN_N      = 100            # sufficiency bar (Decision 4)
HEADER_ROW = 3             # the 'Data' sheet's real header is the 4th row (0-indexed 3)

assert os.path.exists(SRC_XLSX), SRC_XLSX
print("source:", os.path.basename(SRC_XLSX))
print(f"config: year={YEAR}, type={TYPE}, sufficiency n>={MIN_N}")

source: tts_state_download_data_2026_release.xlsx
config: year=2025, type=Residential, sufficiency n>=100


In [2]:
# Load LBNL's published state-median table and filter to residential, target year (Decisions 1-3)
raw = pd.read_excel(SRC_XLSX, sheet_name="Data", header=HEADER_ROW)
raw.columns = [str(c).strip() for c in raw.columns]

med = (
    raw[(raw["type"] == TYPE) & (raw["year"] == YEAR)]
    [["state", "price_dollar_per_watt_median", "price_N"]]
    .rename(columns={"price_dollar_per_watt_median": "lbnl_median_per_w", "price_N": "n"})
    .reset_index(drop=True)
)
med["n"] = pd.to_numeric(med["n"], errors="coerce")

national = float(med.loc[med["state"] == "National", "lbnl_median_per_w"].iloc[0])
print(f"{len(med)} rows ({TYPE}, {YEAR}) incl. National")
print(f"National {YEAR} residential median: ${national:.4f}/W")

52 rows (Residential, 2025) incl. National
National 2025 residential median: $3.6159/W


In [3]:
# EVIDENCE / verification: these must match LBNL's published & viz-tool values exactly.
# (Confirms Decision 1 -- the published table reproduces the headline numbers the row-level file could not.)
_targets = {"NY": 3.87, "NJ": 3.74, "CA": 3.34, "National": 3.62}
print("verification vs LBNL published values:")
for s, t in _targets.items():
    v = float(med.loc[med["state"] == s, "lbnl_median_per_w"].iloc[0])
    print(f"  {s:9} table=${v:.4f}  published=${t}  [{'OK' if abs(round(v,2)-t) < 0.005 else 'MISMATCH'}]")

verification vs LBNL published values:
  NY        table=$3.8746  published=$3.87  [OK]
  NJ        table=$3.7435  published=$3.74  [OK]
  CA        table=$3.3392  published=$3.34  [OK]
  National  table=$3.6159  published=$3.62  [OK]


In [4]:
# Build the baseline upfront-cost table: own published median if n >= MIN_N, else National (Decisions 4-5)
states = med[med["state"] != "National"].copy()
states["sufficient"] = states["n"] >= MIN_N
states["upfront_cost_per_w"] = np.where(states["sufficient"], states["lbnl_median_per_w"], national).round(4)
states["source"] = np.where(
    states["sufficient"], f"LBNL 2025 state median (n>={MIN_N})", "national median (insufficient data)"
)

nat_row = pd.DataFrame([{
    "state": "National", "upfront_cost_per_w": round(national, 4),
    "lbnl_median_per_w": national, "n": float(med.loc[med.state == "National", "n"].iloc[0]),
    "sufficient": True, "source": "LBNL 2025 national median",
}])

out = (pd.concat([states, nat_row], ignore_index=True)
       [["state", "upfront_cost_per_w", "source", "n", "lbnl_median_per_w"]]
       .sort_values("state").reset_index(drop=True))

n_own = int(states["sufficient"].sum())
print(f"{n_own} states use their own median (n>={MIN_N}); {len(states)-n_own} use the national ${national:.2f}/W fallback.")
print("\nstates using their own median (sorted):")
print(out[out.source.str.startswith('LBNL 2025 state')][["state","upfront_cost_per_w","n"]]
      .sort_values("upfront_cost_per_w", ascending=False).to_string(index=False))
print("\npublished-but-insufficient (fall to national), for the record:")
print(states[~states.sufficient & states.n.notna()][["state","lbnl_median_per_w","n"]].to_string(index=False))

13 states use their own median (n>=100); 38 use the national $3.62/W fallback.

states using their own median (sorted):
state  upfront_cost_per_w       n
   NM              4.1410  1682.0
   NC              4.0961  4169.0
   TX              4.0434  2673.0
   NY              3.8746  9031.0
   WI              3.8516  1895.0
   NJ              3.7435  6603.0
   RI              3.5133   215.0
   OR              3.3782  2826.0
   CA              3.3392 23691.0
   CT              3.2803  4080.0
   FL              3.1414   304.0
   AZ              3.1301  3558.0
   WA              3.0389   886.0

published-but-insufficient (fall to national), for the record:
state  lbnl_median_per_w    n
   MA             3.3674 26.0
   MD             3.9524 29.0
   NH             3.6510 43.0


In [5]:
out.to_csv(OUT_CSV, index=False)
print("Saved", os.path.basename(OUT_CSV), "|", len(out), "rows")
out

Saved state_upfront_cost_lbnl_2025.csv | 52 rows


,state,upfront_cost_per_w,source,n,lbnl_median_per_w
0,AK,3.6159,national median (insufficient data),NaN,NaN
1,AL,3.6159,national median (insufficient data),NaN,NaN
2,AR,3.6159,national median (insufficient data),NaN,NaN
3,AZ,3.1301,LBNL 2025 state median (n>=100),3558.0,3.1301
4,CA,3.3392,LBNL 2025 state median (n>=100),23691.0,3.3392
5,CO,3.6159,national median (insufficient data),NaN,NaN
6,CT,3.2803,LBNL 2025 state median (n>=100),4080.0,3.2803
7,DC,3.6159,national median (insufficient data),NaN,NaN
8,DE,3.6159,national median (insufficient data),NaN,NaN
9,FL,3.1414,LBNL 2025 state median (n>=100),304.0,3.1414


## Downstream integration & the rest of the dGen pipeline

`adjust_pv_batt_price_trajectories.ipynb` → `make_state_specific()` reads this CSV, drops the
`National` row, maps `state -> upfront_cost_per_w`, and for each state scales the national **ATB
mid-case decline shape** so the **2026** capex equals that state's `upfront_cost_per_w * 1000`
($/kW). States absent from the map fall back to the national value there too. Only
`system_capex_per_kw` is re-scaled — O&M, battery $/kWh, and `linear_constant` stay national.

**Why BOTH price tables are made state-specific (evidence):** in `financial_functions.py`, the
cashflow cost `system_costs = costs['system_capex_per_kw_combined'] * kw` (lines ~210 / 217 / 264)
uses the **`*_combined`** value, which comes from the **`pv_plus_batt`** table — for the PV-only
pass too. The plain `system_capex_per_kw` (from `pv_price`) is only carried into the reporting dict
(line ~430). So the number that actually drives **PV-only payback → adoption** *and* battery
economics is the pv_plus_batt one. Therefore `make_state_specific()` is applied to **both** `pv`
(→ `pv_price_baseline`, reporting) and `pv_batt` (→ `pv_plus_batt_baseline`, economics) with
identical per-state PV anchors.

`agent_mutation.elec.apply_pv_prices` / `apply_pv_plus_batt_prices` merge on
`['state_abbr', 'year', 'sector_abbr']` when a `state_abbr` column is present (state-specific
baseline tables), and fall back to `['year','sector_abbr']` for national tables (the policy
`*_dollar_per_watt` trajectory is unchanged: immediate $1/W).

### Decision log (quick reference)
1. **Published xlsx, not row-level file** — file is a subset (NJ 100% redacted; CT-2024 2,742 vs 3,663; 27 states only).
2. **PV-only / host-owned / non-self-installed, gross $/Wdc** — LBNL definition; including storage flattens the real decline.
3. **Year 2025** — most recent, LBNL-confirmed complete; National $3.62.
4. **n ≥ 100 sufficiency** — MD/NH/MA (n=29/43/26) fall to national.
5. **National fallback $3.62** — for all insufficient/no-data states (38 of 51).
6. **Battery $/kWh stays 2024 ($1,199.3)** — no published 2025 storage price; row-level 2025 too noisy.
7. **Capacity is DC nameplate ($/Wdc)** — matches how dGen sizes/prices PV.

To rebuild: run this notebook (produces the CSV), then run `adjust_pv_batt_price_trajectories.ipynb`
with the Cloud SQL proxy up (uploads the state-keyed baseline tables), then launch the baseline run.